In [1]:
try:
    import google.colab  # noqa: F401

    %pip install -q dataeval maite-datasets
except Exception:
    pass

In [2]:
import numpy as np
import polars as pl
import torch
from IPython.display import display
from maite_datasets.object_detection import VOCDetection
from torchvision.models import ResNet18_Weights, resnet18
from torchvision.transforms.v2 import GaussianNoise

from dataeval import Embeddings, Metadata
from dataeval.config import set_device, set_seed
from dataeval.core import label_parity
from dataeval.data import Relabel, View
from dataeval.extractors import TorchExtractor
from dataeval.shift import ChunkedDrift, DriftDomainClassifier, DriftKNeighbors, DriftMMD, DriftUnivariate

# Set a random seed
rng = np.random.default_rng(213)

# Set default device for notebook
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_device(device)

# Seed NumPy and torch so the added Gaussian noise (and drift detectors) are reproducible
set_seed(213, all_generators=True)

In [3]:
# Load the training dataset
train_ds = VOCDetection("./data", year="2012", image_set="train", download=True)
print(train_ds)
print(f"Image 0 shape: {train_ds[0][0].shape}")

VOCDetection Dataset
--------------------
    Year: 2012
    Transforms: []
    Image Set: train
    Metadata: {'id': 'VOCDetection_train', 'index2label': {0: 'aeroplane', 1: 'bicycle', 2: 'bird', 3: 'boat', 4: 'bottle', 5: 'bus', 6: 'car', 7: 'cat', 8: 'chair', 9: 'cow', 10: 'diningtable', 11: 'dog', 12: 'horse', 13: 'motorbike', 14: 'person', 15: 'pottedplant', 16: 'sheep', 17: 'sofa', 18: 'train', 19: 'tvmonitor'}, 'split': 'train'}
    Path: /builds/jatic/aria/dataeval/docs/source/notebooks/data/vocdataset/VOCdevkit/VOC2012
    Size: 5717
Image 0 shape: (3, 442, 500)


In [4]:
# Load the "operational" dataset
operational_ds = VOCDetection("./data", year="2012", image_set="val", download=True)
print(operational_ds)
print(f"Image 0 shape: {operational_ds[0][0].shape}")

VOCDetection Dataset
--------------------
    Year: 2012
    Transforms: []
    Image Set: val
    Metadata: {'id': 'VOCDetection_val', 'index2label': {0: 'aeroplane', 1: 'bicycle', 2: 'bird', 3: 'boat', 4: 'bottle', 5: 'bus', 6: 'car', 7: 'cat', 8: 'chair', 9: 'cow', 10: 'diningtable', 11: 'dog', 12: 'horse', 13: 'motorbike', 14: 'person', 15: 'pottedplant', 16: 'sheep', 17: 'sofa', 18: 'train', 19: 'tvmonitor'}, 'split': 'val'}
    Path: /builds/jatic/aria/dataeval/docs/source/notebooks/data/vocdataset/VOCdevkit/VOC2012
    Size: 5823
Image 0 shape: (3, 375, 500)


In [5]:
# The indoor concepts this deployment monitors
FURNITURE = ("chair", "diningtable", "sofa", "tvmonitor")

# Keep these four concepts as-is; every other VOC class is dropped as out-of-vocabulary
furniture_only = Relabel({name: name for name in FURNITURE}, FURNITURE)

train_view = View(train_ds, operations=[furniture_only])
operational_view = View(operational_ds, operations=[furniture_only])

print(f"train:       {len(train_ds)} -> {len(train_view)} images")
print(f"operational: {len(operational_ds)} -> {len(operational_view)} images")
print(f"vocabulary:  {train_view.metadata.get('index2label')}")

train:       5717 -> 1163 images
operational: 5823 -> 1173 images
vocabulary:  {0: 'chair', 1: 'diningtable', 2: 'sofa', 3: 'tvmonitor'}


In [6]:
resnet = resnet18(weights=ResNet18_Weights.DEFAULT, progress=False)
transforms = ResNet18_Weights.DEFAULT.transforms()
extractor = TorchExtractor(resnet, transforms=transforms, layer_name="avgpool")

# Create embeddings for the train and operational splits
train_embs = Embeddings(train_view, extractor=extractor, batch_size=64)
operational_embs = Embeddings(operational_view, extractor=extractor, batch_size=64)

In [7]:
print(f"({len(train_embs)}, {train_embs[0].shape})")  # (1163, shape)
print(f"({len(operational_embs)}, {operational_embs[0].shape})")  # (1173, shape)

(1163, (512,))
(1173, (512,))


In [8]:
# A type alias for all of the drift detectors
DriftDetector = DriftUnivariate | DriftMMD | DriftDomainClassifier | DriftKNeighbors

# Create a mapping for the detectors to iterate over
detectors: dict[str, DriftDetector] = {
    "CVM": DriftUnivariate(method="cvm").fit(train_embs),
    "MMD": DriftMMD().fit(train_embs),
    "MVDC": DriftDomainClassifier().fit(train_embs),
    "KNN": DriftKNeighbors().fit(train_embs),
}

In [9]:
# Iterate and print the name of the detector class and its boolean drift prediction
clean_results = {name: detector.predict(operational_embs) for name, detector in detectors.items()}

print("\n".join(f"{res[0]} detected drift? {res[1].drifted}" for res in clean_results.items()))

CVM detected drift? False
MMD detected drift? False
MVDC detected drift? False
KNN detected drift? False


In [10]:
# Define transform with added gaussian noise
noisy_transforms = [transforms, GaussianNoise()]

# Create extractor with noisy transforms
noisy_extractor = TorchExtractor(resnet, transforms=noisy_transforms, layer_name="avgpool")

# Applies gaussian noise to images before processing
noisy_embs = Embeddings(operational_view, extractor=noisy_extractor, batch_size=64)

In [11]:
# Iterate and print the name of the detector class and its boolean drift prediction
print("\n".join(f"{det[0]} detected drift? {det[1].predict(noisy_embs).drifted}" for det in detectors.items()))

CVM detected drift? True
MMD detected drift? True
MVDC detected drift? True
KNN detected drift? True


In [12]:
# Store results for inspection
results = {name: detector.predict(noisy_embs) for name, detector in detectors.items()}

In [13]:
cvm_result = results["CVM"]
cvm_details = cvm_result.details

n_drifted = sum(cvm_details["feature_drift"])
n_features = len(cvm_details["feature_drift"])
print(f"Features drifted: {n_drifted}/{n_features}")
print(f"Corrected p-value threshold: {cvm_details['feature_threshold']:.6f}")
print(f"Min feature p-value: {min(cvm_details['p_vals']):.6f}")
print(f"Max feature p-value: {max(cvm_details['p_vals']):.6f}")

Features drifted: 474/512
Corrected p-value threshold: 0.050000
Min feature p-value: 0.000000
Max feature p-value: 0.937497


In [14]:
mvdc_result = results["MVDC"]
mvdc_details = mvdc_result.details

print(f"AUROC: {mvdc_result.distance:.4f} (threshold: {mvdc_result.threshold})")
print(f"Per-fold AUROCs: {[round(a, 4) for a in mvdc_details['fold_aurocs']]}")

# Show top 5 most important features
importances = np.array(mvdc_details["feature_importances"])
top_indices = np.argsort(importances)[::-1][:5]
print("\nTop 5 features driving drift:")
print("\n".join(f"  Feature {idx}: importance = {importances[idx]:.4f}" for idx in top_indices))

AUROC: 0.9997 (threshold: 0.55)
Per-fold AUROCs: [np.float32(0.9994), np.float32(0.999), np.float32(0.9997), np.float32(1.0), np.float32(0.9998)]

Top 5 features driving drift:
  Feature 104: importance = 298.6000
  Feature 142: importance = 207.0000
  Feature 243: importance = 158.0000
  Feature 221: importance = 137.8000
  Feature 47: importance = 102.2000


In [15]:
knn_result = results["KNN"]
knn_details = knn_result.details

print(f"Mean reference k-NN distance: {knn_details['mean_ref_distance']:.4f}")
print(f"Mean test k-NN distance:      {knn_details['mean_test_distance']:.4f}")
print(f"Distance increase:             {knn_details['mean_test_distance'] - knn_details['mean_ref_distance']:.4f}")
print(f"P-value:                       {knn_details['p_val']:.6f}")

Mean reference k-NN distance: 18.4039
Mean test k-NN distance:      20.2793
Distance increase:             1.8754
P-value:                       0.000000


In [16]:
mmd_result = results["MMD"]
mmd_details = mmd_result.details

print(f"MMD² distance:   {mmd_result.distance:.6f}")
print(f"MMD² threshold:  {mmd_details['distance_threshold']:.6f}")
print(f"P-value:         {mmd_details['p_val']:.6f}")

MMD² distance:   0.129441
MMD² threshold:  0.000299
P-value:         0.000000


In [17]:
# Build a combined array: first 40% clean, last 60% noisy
n_operational = len(operational_embs)
split_idx = int(n_operational * 0.4)

combined_embs = np.concatenate([operational_embs[:split_idx], noisy_embs[split_idx:]])
print(f"Combined shape: {combined_embs.shape} (clean: {split_idx}, noisy: {n_operational - split_idx})")

Combined shape: (1173, 512) (clean: 469, noisy: 704)


In [18]:
# Re-fit detectors with chunking enabled (5 chunks each)
chunked_detectors: dict[str, ChunkedDrift] = {
    "CVM": DriftUnivariate(method="cvm").chunked(chunk_count=5).fit(train_embs),
    "MMD": DriftMMD().chunked(chunk_count=5).fit(train_embs),
    "MVDC": DriftDomainClassifier(threshold=(0.45, 0.65)).chunked(chunk_count=5).fit(train_embs),
    "KNN": DriftKNeighbors().chunked(chunk_count=5).fit(train_embs),
}

In [19]:
chunked_results = {}

for name, detector in chunked_detectors.items():
    result = detector.predict(combined_embs)
    chunked_results[name] = result
    print(f"\n{name} - Overall drift detected: {result.drifted} (metric: {result.metric_name})")
    if isinstance(result.details, pl.DataFrame):
        display(result.details)


CVM - Overall drift detected: True (metric: cvm_distance)


key,index,start_index,end_index,value,upper_threshold,lower_threshold,drifted
str,i64,i64,i64,f64,f64,f64,bool
"""[0:232]""",0,0,232,0.200583,0.283279,0.124827,false
"""[233:465]""",1,233,465,0.165294,0.283279,0.124827,false
"""[466:698]""",2,466,698,5.352722,0.283279,0.124827,true
"""[699:931]""",3,699,931,5.577215,0.283279,0.124827,true
"""[932:1172]""",4,932,1172,5.403076,0.283279,0.124827,true



MMD - Overall drift detected: True (metric: mmd2)


key,index,start_index,end_index,value,upper_threshold,lower_threshold,drifted
str,i64,i64,i64,f64,f64,f64,bool
"""[0:232]""",0,0,232,0.00136,0.003294,-0.00096,false
"""[233:465]""",1,233,465,0.000149,0.003294,-0.00096,false
"""[466:698]""",2,466,698,0.129371,0.003294,-0.00096,true
"""[699:931]""",3,699,931,0.137718,0.003294,-0.00096,true
"""[932:1172]""",4,932,1172,0.126835,0.003294,-0.00096,true



MVDC - Overall drift detected: True (metric: auroc)


key,index,start_index,end_index,value,upper_threshold,lower_threshold,drifted
str,i64,i64,i64,f64,f64,f64,bool
"""[0:232]""",0,0,232,0.497658,0.65,0.45,false
"""[233:465]""",1,233,465,0.45374,0.65,0.45,false
"""[466:698]""",2,466,698,0.994129,0.65,0.45,true
"""[699:931]""",3,699,931,0.999646,0.65,0.45,true
"""[932:1172]""",4,932,1172,0.999429,0.65,0.45,true



KNN - Overall drift detected: True (metric: knn_distance)


key,index,start_index,end_index,value,upper_threshold,lower_threshold,drifted
str,i64,i64,i64,f64,f64,f64,bool
"""[0:232]""",0,0,232,18.088791,19.212898,17.59515,false
"""[233:465]""",1,233,465,18.360954,19.212898,17.59515,false
"""[466:698]""",2,466,698,20.383005,19.212898,17.59515,true
"""[699:931]""",3,699,931,20.451269,19.212898,17.59515,true
"""[932:1172]""",4,932,1172,20.208296,19.212898,17.59515,true


In [20]:
# Get the metadata for each view
train_md = Metadata(train_view)
operational_md = Metadata(operational_view)

# The views expose the four monitored classes
label_parity(train_md.class_labels, operational_md.class_labels, num_classes=len(FURNITURE))["p_value"]

/tmp/ipykernel_7468/4081706760.py:6: UserWarning: `filename` and `mask_path` were dropped: nearly every row holds a different value, so the column identifies rows rather than grouping them, and it is not numeric so there is no order along which to cut it into groups. Map the values onto a smaller vocabulary to keep the factor. See Metadata.dropped_factors.
  label_parity(train_md.class_labels, operational_md.class_labels, num_classes=len(FURNITURE))["p_value"]


0.9581780599352742

In [21]:
# An upstream policy stops distinguishing sofas from chairs -- the images are untouched
policy_change = Relabel(
    {"chair": "chair", "diningtable": "diningtable", "sofa": "chair", "tvmonitor": "tvmonitor"},
    FURNITURE,
)

# Nest the change on the operational view rather than rebuilding it from the raw split
poor_parity_view = View(operational_view, operations=[policy_change])
poor_parity_md = Metadata(poor_parity_view)

# Gather the counts of each label in the training and broken operational sets
train_label_counts = np.bincount(np.asarray(train_md.class_labels), minlength=len(FURNITURE))
poor_parity_label_counts = np.bincount(np.asarray(poor_parity_md.class_labels), minlength=len(FURNITURE))

print(f"images:                   {len(operational_view)} -> {len(poor_parity_view)} (unchanged)")
print(f"train label counts:       {train_label_counts}")
print(f"poor parity label counts: {poor_parity_label_counts}")

images:                   1173 -> 1173 (unchanged)
train label counts:       [1457  373  399  412]
poor parity label counts: [1836  374    0  414]


/tmp/ipykernel_7468/509078241.py:13: UserWarning: `filename` and `mask_path` were dropped: nearly every row holds a different value, so the column identifies rows rather than grouping them, and it is not numeric so there is no order along which to cut it into groups. Map the values onto a smaller vocabulary to keep the factor. See Metadata.dropped_factors.
  poor_parity_label_counts = np.bincount(np.asarray(poor_parity_md.class_labels), minlength=len(FURNITURE))


In [22]:
label_parity(train_md.class_labels, poor_parity_md.class_labels, num_classes=len(FURNITURE))["p_value"]

3.343376493409642e-108

In [23]:
# TEST ASSERTION CELL ###
# Lock the claims this tutorial makes in prose, so a dependency or data change cannot
# silently invert them while the narrative keeps asserting the old result.

# The monitored slice is the size the prose and inline comments quote
assert len(train_view) == 1163, f"train view size changed: {len(train_view)}"
assert len(operational_view) == 1173, f"operational view size changed: {len(operational_view)}"
assert tuple(train_view.metadata.get("index2label", {}).values()) == FURNITURE

# "There is no drift detected between the train and operational embeddings"
for name, result in clean_results.items():
    assert not result.drifted, f"{name} reported drift on clean operational data"

# "Now drift is detected!" -- every detector must fire once noise is added
for name, result in results.items():
    assert result.drifted, f"{name} failed to detect drift on noisy data"

# "The first two chunks (covering the clean 40%) should show no drift, while the
# later chunks (covering the noisy 60%) should trigger drift alerts."
for name, result in chunked_results.items():
    assert list(result.details["drifted"]) == [False, False, True, True, True], (
        f"{name} chunk pattern changed: {list(result.details['drifted'])}"
    )

# "a p_value of ~0.96 ... close to 1.0"
parity_p_value = label_parity(train_md.class_labels, operational_md.class_labels, num_classes=len(FURNITURE))["p_value"]
assert parity_p_value > 0.9, f"label parity dropped to {parity_p_value}"

# "The p_value is now effectively zero" -- and the contrived set must change labels only
assert len(poor_parity_view) == len(operational_view), "the policy change should not drop images"
assert len(poor_parity_md.class_labels) == len(operational_md.class_labels), "it should not drop annotations"
poor_parity_result = label_parity(train_md.class_labels, poor_parity_md.class_labels, num_classes=len(FURNITURE))
poor_parity_p_value = poor_parity_result["p_value"]
assert poor_parity_p_value < 0.01, f"contrived poor-parity set no longer trips the check: {poor_parity_p_value}"